# Table 2 — Poisson Difference-in-Differences: Patent Counts on Topic Relevance × Post-2018

| Item | Description |
|---|---|
| Goal | Did firms whose 10-K business description is closer to the consumer data privacy debate change their patenting after the Facebook–Cambridge Analytica revelations of March 2018, relative to less relevant firms in the same NAICS two-digit code and year? Specification: `ppmlhdfe patent_count c.topic_relevance##Post controls, absorb(naics2#fyear) vce(cluster firm_id)`, with Post equal to one for fiscal years 2018 to 2021; `##` enters both main effects and the interaction, and `ppmlhdfe` drops the main effects that the fixed effects absorb. |
| Comparison | Continuous difference-in-differences: the change from the pre-period (fiscal 1999 to 2017) to the post-period (2018 to 2021) in the yearly patent count of firms with a higher relevance score, against the same change for firms with a lower score, within NAICS two-digit-by-year cells. |
| Unit and sample | Firm-year: the T4.1.3 analysis sample (689 firms with at least one patent filed in 1999 to 2017) in fiscal 1999 to 2021 with a 10-K and the three lagged controls (T4.1.5; 11,406 firm-years, 685 firms). `ppmlhdfe` drops cells or firms whose count is zero throughout; the surviving observations and firms are reported per column. |
| Treated group / exposure | Continuous: the firm's topic-relevance score (T3.8.3, one score per firm), standardized to mean zero and standard deviation one over the regression firm-years, interacted with Post. There is no binary treated group. |
| Control / reference group | Lower-score firms in the same NAICS two-digit code and year; the pre-period is the baseline. Controls at t − 1: log firm age, log total assets, log of one plus R&D expense with a missing flag. Columns 2 and 4 add firm fixed effects, which absorb the score's main effect and leave the interaction identified from within-firm changes. |
| Timing and baseline | Pre-period fiscal 1999 to 2017, post-period 2018 to 2021; the Post main effect is absorbed by the NAICS-by-year fixed effects; the count of year t is the count filed in calendar year t. |
| Outcome | Columns 1 and 2: KPSS patents filed in the year with at least one code of the privacy code list (T4.1.2). Columns 3 and 4: all KPSS patents filed in the year, as a comparison outcome. |
| Estimate and interpretation | The coefficient on Topic relevance × Post is the post-minus-pre change in the log expected count per standard deviation of relevance, relative to less relevant firms; exp(coefficient) minus one is the percent change. A positive value means more relevant firms raised their patenting after 2018 relative to their peers. |
| Adjustment and inference | `ppmlhdfe` with NAICS two-digit-by-year fixed effects (columns 1 and 3) or NAICS two-digit-by-year and firm fixed effects (columns 2 and 4); standard errors clustered by firm. |
| Assumption and limits | Parallel trends in logs: absent the 2018 events, the patent counts of more and less relevant firms would have moved in parallel within NAICS-year. The companion event-study notebook (T4.1.X - Figure) plots the year-by-year interactions with 2017 as the base to check this. The post-period is four years, and KPSS covers granted patents only, so filings late in the window are undercounted for every firm; the fixed effects absorb this only if the undercount does not vary with relevance. |
| Comparison status | Specification requested by the user, with Post defined as fiscal 2018 onward (the Cambridge Analytica revelations of March 2018). Added: the firm-fixed-effect columns and the all-patents outcome. Open: the choice of 2018 over 2017 (the first GDPR and CCPA year) as the break. |
| Input | `Processed/privacy_regression_panel.dta` from T4.1.5. |
| Output | Table 2 as Markdown in this notebook and as `table_2_ppml_did_patents_relevance_post2018.{tex,csv,md}` in `output/T4.1.X`. |


## Step 1. Set the Stata session options

A fresh session: the Stata version, no leftover data or estimates, no paging, and a wide line.


In [0]:
version 18.5
clear all
set more off
set linesize 255


## Step 2. Define inputs and output locations

The two roots and the S5 folders as globals, the post-period start, and the control list.


In [1]:
global PROJECT_FOLDER "/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis"
global SCRATCH_FOLDER "/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis"
global NOTEBOOK_SECTION "T4 - Toy Example [Topic Relevance on Privacy Innovation]"
global NOTEBOOK_SUBFOLDER "${NOTEBOOK_SECTION}/T4.1 - Privacy Patents and Topic Relevance"
global NOTEBOOK_NAME "T4.1.X"
global PROCESSED_FOLDER "${SCRATCH_FOLDER}/${NOTEBOOK_SUBFOLDER}/Processed"
global TABLE_FOLDER "${PROJECT_FOLDER}/${NOTEBOOK_SECTION}/output/${NOTEBOOK_NAME}"
global PANEL_FILE "${PROCESSED_FOLDER}/privacy_regression_panel.dta"
global POST_START 2018
global CONTROLS "ln_age_l1 ln_at_l1 ln_xrd_l1 rd_missing_l1"
capture mkdir "${TABLE_FOLDER}"
display "Panel: ${PANEL_FILE}"
display "Tables: ${TABLE_FOLDER}"


Panel: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T4 - Toy Example [Topic Relevance on Privacy Innovation]/T4.1 - Privacy Patents and Topic Relevance/Processed/privacy_regression_panel.dta
Tables: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T4 - Toy Example [Topic Relevance on Privacy Innovation]/output/T4.1.X


## Step 3. Load the panel and check the observation unit

One row per firm and fiscal year; the first rows are shown with the variables the regressions use.


In [2]:
use "${PANEL_FILE}", clear
isid gvkey fyear
count
list gvkey fyear naics2 n_privacy_patents n_patents_all relevance_firm ln_age_l1 ln_at_l1 ln_xrd_l1 rd_missing_l1 in 1/5, abbreviate(18) noobs


  11,464

  +----------------------------------------------------------------------------------------------------------------------------------+
  |  gvkey   fyear   naics2   n_privacy_patents   n_patents_all   relevance_firm   ln_age_l1    ln_at_l1   ln_xrd_l1   rd_missing_l1 |
  |----------------------------------------------------------------------------------------------------------------------------------|
  | 001013    1999       33                   1              60          .522551   2.9957323    7.170571   4.9338406               0 |
  | 001013    2000       33                   1              80          .522551   3.0445224   7.4220921   5.6435794               0 |
  | 001013    2001       33                   0             131          .522551   3.0910425   8.2866473   5.8260001               0 |
  | 001013    2002       33                   0              95          .522551   3.1354942    7.823926     5.63336               0 |
  | 001013    2003       33                  

## Step 4. Restrict the sample and define the regression variables

Firm-years with all three lagged controls. The score is standardized over these firm-years; Post marks fiscal years from the post-period start; the firm id and the NAICS two-digit-by-year cell id are built. The pre- and post-period sizes and a summary of the regression variables are reported.


In [3]:
drop if missing(relevance_firm, ln_age_l1, ln_at_l1, ln_xrd_l1)
count
egen firm_id = group(gvkey)
egen naics2_year = group(naics2 fyear)
egen relevance_z = std(relevance_firm)
gen byte post = fyear >= ${POST_START}
label variable relevance_z "Firm-level topic relevance, standardized over the regression firm-years"
label variable post "Fiscal year ${POST_START} or later"
egen firm_tag = tag(gvkey)
quietly count if firm_tag
display "Firms: " r(N)
tab post
summarize n_privacy_patents n_patents_all relevance_z ${CONTROLS}


(58 observations deleted)
  11,406


Firms: 685

Fiscal year |
    2018 or |
      later |      Freq.     Percent        Cum.
------------+-----------------------------------
          0 |      9,606       84.22       84.22
          1 |      1,800       15.78      100.00
------------+-----------------------------------
      Total |     11,406      100.00

    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
n_privacy_~s |     11,406    2.827547    17.85006          0        586
n_patents_~l |     11,406    101.7283    390.6174          0       9868
 relevance_z |     11,406    6.70e-10           1  -3.113457    2.98371
   ln_age_l1 |     11,406    3.121181    .9266316          0   4.276666
    ln_at_l1 |     11,406    9.000192    1.712154   2.067369   15.19824
-------------+---------------------------------------------------------
   ln_xrd_l1 |     11,406    3.038199    2.843513          0   10.66291
rd_missing~1 |     11,406    

## Step 5. Column (1): privacy patents with NAICS two-digit-by-year fixed effects, the requested specification

What this cell estimates: the post-minus-pre change in the log expected count of privacy patents per standard deviation of the firm's topic relevance, relative to less relevant firms in the same NAICS two-digit code and year, given the lagged controls. With the `##` notation Stata adds both main effects; Post's is absorbed by the NAICS-by-year cells and dropped, and the score's main effect is the pre-period cross-firm difference. The identifying assumption is parallel trends in logs across relevance before 2018; the companion event-study notebook checks it year by year. Standard errors are clustered by firm. Units with a zero count throughout drop out of the Poisson likelihood; the surviving observations and firms are kept for the table.


In [4]:
eststo clear
ppmlhdfe n_privacy_patents c.relevance_z##i.post ${CONTROLS}, absorb(naics2_year) vce(cluster firm_id)
summarize `e(depvar)' if e(sample)
estadd scalar mean_dv = r(mean)
estadd scalar clusters = e(N_clust)
estadd local controls "Yes"
estadd local firm_fe "No"
estadd local naics2_year_fe "Yes"
estadd local dv_label "Privacy patents"
eststo col_1


(dropped 2096 observations that are either singletons or separated by a fixed effect)


note: 1 variable omitted because of collinearity: 1bn.post


Iteration 1:   deviance = 6.6856e+04  eps = .         iters = 1    tol = 1.0e-04  min(eta) =  -6.54  P   


Iteration 2:   deviance = 5.0290e+04  eps = 3.29e-01  iters = 1    tol = 1.0e-04  min(eta) =  -7.88      


Iteration 3:   deviance = 4.7770e+04  eps = 5.28e-02  iters = 1    tol = 1.0e-04  min(eta) =  -9.36      


Iteration 4:   deviance = 4.7425e+04  eps = 7.26e-03  iters = 1    tol = 1.0e-04  min(eta) = -10.43      


... (12 outputs left out)


Absorbed degrees of freedom:
-----------------------------------------------------+
 Absorbed FE | Categories  - Redundant  = Num. Coefs |


-------------+---------------------------------------|
 naics2_year |       249           0         249     |


-----------------------------------------------------+



    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
n_privacy_~s |      9,310    3.454672    19.70093          0        586

added scalar:
            e(mean_dv) =  3.4546724

added scalar:
           e(clusters) =  652

added macro:
           e(controls) : "Yes"

added macro:
            e(firm_fe) : "No"

added macro:
     e(naics2_year_fe) : "Yes"

added macro:
           e(dv_label) : "Privacy patents"


## Step 6. Column (2): privacy patents adding firm fixed effects

What this cell estimates: the post-minus-pre change in the log expected count of privacy patents per standard deviation of the firm's topic relevance, relative to less relevant firms in the same NAICS two-digit code and year, given the lagged controls. With the `##` notation Stata adds both main effects; Post's is absorbed by the NAICS-by-year cells and the score's by the firm fixed effects, so ppmlhdfe drops both and only the interaction is estimated. The identifying assumption is parallel trends in logs across relevance before 2018; the companion event-study notebook checks it year by year. Standard errors are clustered by firm. Units with a zero count throughout drop out of the Poisson likelihood; the surviving observations and firms are kept for the table.


In [5]:
ppmlhdfe n_privacy_patents c.relevance_z##i.post ${CONTROLS}, absorb(naics2_year firm_id) vce(cluster firm_id)
summarize `e(depvar)' if e(sample)
estadd scalar mean_dv = r(mean)
estadd scalar clusters = e(N_clust)
estadd local controls "Yes"
estadd local firm_fe "Yes"
estadd local naics2_year_fe "Yes"
estadd local dv_label "Privacy patents"
eststo col_2


(dropped 6940 observations that are either singletons or separated by a fixed effect)


note: 2 variables omitted because of collinearity: relevance_z 1bn.post


 $$ Stopping (no negative residuals); separation found in 0 observations (1 iterations and 18 subiterations)


Iteration 1:   deviance = 1.6982e+04  eps = .         iters = 6    tol = 1.0e-04  min(eta) =  -4.12  P   


Iteration 2:   deviance = 1.0455e+04  eps = 6.24e-01  iters = 5    tol = 1.0e-04  min(eta) =  -5.61      


Iteration 3:   deviance = 9.2635e+03  eps = 1.29e-01  iters = 5    tol = 1.0e-04  min(eta) =  -7.04      


... (4 outputs left out)

Iteration 8:   deviance = 9.0233e+03  eps = 8.74e-10  iters = 3    tol = 1.0e-05  min(eta) =  -9.37   S  


Iteration 9:   deviance = 9.0233e+03  eps = 1.94e-15  iters = 3    tol = 1.0e-07  min(eta) =  -9.37   S  
Iteration 10:  deviance = 9.0233e+03  eps = 1.76e-16  iters = 2    tol = 1.0e-09  min(eta) =  -9.37   S O
------------------------------------------------------------------------------------------------------------
(legend: p: exact partial-out   s: exact solver   h: step-halving   o: epsilon below tolerance)
Converged in 10 iterations and 38 HDFE sub-iterations (tol = 1.0e-08)



HDFE PPML regression                              No. of obs      =      4,466
Absorbing 2 HDFE groups                           Residual df     =        257
Statistics robust to heteroskedasticity           Wald chi2(5)    =      21.24
Deviance             =  9023.322516               Prob > chi2     =     0.0007
Log pseudolikelihood = -7934.939887               Pseudo R2       =     0.8845

Number of clusters (firm_id)=        258
                                    (Std. err. adjusted for 258 clusters in firm_id)
------------------------------------------------------------------------------------
                   |               Robust
 n_privacy_patents | Coefficient  std. err.      z    P>|z|     [95% conf. interval]
-------------------+----------------------------------------------------------------
       relevance_z |          0  (omitted)
            1.post |          0  (omitted)
                   |
post#c.relevance_z |
                1  |    -.09485    .067493    -1.41 

-----------------------------------------------------+
 Absorbed FE | Categories  - Redundant  = Num. Coefs |
-------------+---------------------------------------|
 naics2_year |       239           1         238     |
     firm_id |       258         258           0    *|
-----------------------------------------------------+
* = FE nested within cluster; treated as redundant for DoF computation

    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
n_privacy_~s |      4,466     7.19906    27.96858          0        586

added scalar:
            e(mean_dv) =  7.1990596

added scalar:
           e(clusters) =  258

added macro:
           e(controls) : "Yes"

added macro:
            e(firm_fe) : "Yes"

added macro:
     e(naics2_year_fe) : "Yes"

added macro:
           e(dv_label) : "Privacy patents"


## Step 7. Column (3): all patents with NAICS two-digit-by-year fixed effects

What this cell estimates: the post-minus-pre change in the log expected count of all patents per standard deviation of the firm's topic relevance, relative to less relevant firms in the same NAICS two-digit code and year, given the lagged controls. With the `##` notation Stata adds both main effects; Post's is absorbed by the NAICS-by-year cells and dropped, and the score's main effect is the pre-period cross-firm difference. The identifying assumption is parallel trends in logs across relevance before 2018; the companion event-study notebook checks it year by year. Standard errors are clustered by firm. Units with a zero count throughout drop out of the Poisson likelihood; the surviving observations and firms are kept for the table.


In [6]:
ppmlhdfe n_patents_all c.relevance_z##i.post ${CONTROLS}, absorb(naics2_year) vce(cluster firm_id)
summarize `e(depvar)' if e(sample)
estadd scalar mean_dv = r(mean)
estadd scalar clusters = e(N_clust)
estadd local controls "Yes"
estadd local firm_fe "No"
estadd local naics2_year_fe "Yes"
estadd local dv_label "All patents"
eststo col_3


(dropped 130 observations that are either singletons or separated by a fixed effect)


note: 1 variable omitted because of collinearity: 1bn.post


Iteration 1:   deviance = 1.4212e+06  eps = .         iters = 1    tol = 1.0e-04  min(eta) =  -5.47  P   


Iteration 2:   deviance = 1.1408e+06  eps = 2.46e-01  iters = 1    tol = 1.0e-04  min(eta) =  -6.24      


Iteration 3:   deviance = 1.1107e+06  eps = 2.71e-02  iters = 1    tol = 1.0e-04  min(eta) =  -7.04      


Iteration 4:   deviance = 1.1060e+06  eps = 4.27e-03  iters = 1    tol = 1.0e-04  min(eta) =  -7.83      


... (3 outputs left out)

Iteration 8:   deviance = 1.1050e+06  eps = 4.49e-07  iters = 1    tol = 1.0e-05  min(eta) = -10.45   S  


Iteration 9:   deviance = 1.1050e+06  eps = 4.02e-09  iters = 1    tol = 1.0e-06  min(eta) = -10.50   S O
------------------------------------------------------------------------------------------------------------
(legend: p: exact partial-out   s: exact solver   h: step-halving   o: epsilon below tolerance)
Converged in 9 iterations and 9 HDFE sub-iterations (tol = 1.0e-08)



HDFE PPML regression                              No. of obs      =     11,276
Absorbing 1 HDFE group                            Residual df     =        682
Statistics robust to heteroskedasticity           Wald chi2(6)    =     462.92
Deviance             =  1104994.047               Prob > chi2     =     0.0000
Log pseudolikelihood =   -572122.86               Pseudo R2       =     0.7391

Number of clusters (firm_id)=        683


                                    (Std. err. adjusted for 683 clusters in firm_id)
------------------------------------------------------------------------------------
                   |               Robust
     n_patents_all | Coefficient  std. err.      z    P>|z|     [95% conf. interval]
-------------------+----------------------------------------------------------------
       relevance_z |   .2114607   .1230132     1.72   0.086    -.0296408    .4525622
            1.post |          0  (omitted)
                   |
post#c.relevance_z |
                1  |  -.1240108    .076775    -1.62   0.106     -.274487    .0264654
                   |
         ln_age_l1 |   .2918785   .1719537     1.70   0.090    -.0451445    .6289015
          ln_at_l1 |   .2504335   .0611221     4.10   0.000     .1306363    .3702307
         ln_xrd_l1 |   .5726963   .0891116     6.43   0.000     .3980407    .7473519
     rd_missing_l1 |   1.763046   .5384712     3.27   0.001     .7076624     2.81843
  

## Step 8. Column (4): all patents adding firm fixed effects

What this cell estimates: the post-minus-pre change in the log expected count of all patents per standard deviation of the firm's topic relevance, relative to less relevant firms in the same NAICS two-digit code and year, given the lagged controls. With the `##` notation Stata adds both main effects; Post's is absorbed by the NAICS-by-year cells and the score's by the firm fixed effects, so ppmlhdfe drops both and only the interaction is estimated. The identifying assumption is parallel trends in logs across relevance before 2018; the companion event-study notebook checks it year by year. Standard errors are clustered by firm. Units with a zero count throughout drop out of the Poisson likelihood; the surviving observations and firms are kept for the table.


In [7]:
ppmlhdfe n_patents_all c.relevance_z##i.post ${CONTROLS}, absorb(naics2_year firm_id) vce(cluster firm_id)
summarize `e(depvar)' if e(sample)
estadd scalar mean_dv = r(mean)
estadd scalar clusters = e(N_clust)
estadd local controls "Yes"
estadd local firm_fe "Yes"
estadd local naics2_year_fe "Yes"
estadd local dv_label "All patents"
eststo col_4


(dropped 277 observations that are either singletons or separated by a fixed effect)


note: 2 variables omitted because of collinearity: relevance_z 1bn.post


 $$ Stopping (no negative residuals); separation found in 0 observations (1 iterations and 15 subiterations)


Iteration 1:   deviance = 4.5084e+05  eps = .         iters = 6    tol = 1.0e-04  min(eta) =  -4.03  P   


Iteration 2:   deviance = 2.4489e+05  eps = 8.41e-01  iters = 5    tol = 1.0e-04  min(eta) =  -5.59      


Iteration 3:   deviance = 2.0268e+05  eps = 2.08e-01  iters = 5    tol = 1.0e-04  min(eta) =  -7.12      


... (6 outputs left out)

Iteration 10:  deviance = 1.8815e+05  eps = 7.15e-09  iters = 4    tol = 1.0e-06  min(eta) = -11.79   S  


Iteration 11:  deviance = 1.8815e+05  eps = 1.70e-13  iters = 4    tol = 1.0e-07  min(eta) = -11.79   S  


Iteration 12:  deviance = 1.8815e+05  eps = 1.19e-16  iters = 2    tol = 1.0e-09  min(eta) = -11.79   S O
------------------------------------------------------------------------------------------------------------
(legend: p: exact partial-out   s: exact solver   h: step-halving   o: epsilon below tolerance)
Converged in 12 iterations and 53 HDFE sub-iterations (tol = 1.0e-08)



HDFE PPML regression                              No. of obs      =     11,129
Absorbing 2 HDFE groups                           Residual df     =        654
Statistics robust to heteroskedasticity           Wald chi2(5)    =      91.82
Deviance             =  188147.6186               Prob > chi2     =     0.0000
Log pseudolikelihood =  -113689.828               Pseudo R2       =     0.9478

Number of clusters (firm_id)=        655
                                    (Std. err. adjusted for 655 clusters in firm_id)
------------------------------------------------------------------------------------
                   |               Robust
     n_patents_all | Coefficient  std. err.      z    P>|z|     [95% conf. interval]
-------------------+----------------------------------------------------------------
       relevance_z |          0  (omitted)
            1.post |          0  (omitted)
                   |
post#c.relevance_z |
                1  |  -.0014719   .0517162    -0.03 

## Step 9. Assemble Table 2, display it, and export it

The four stored columns become one S4 table: the dependent variable row, the interaction and main-effect coefficients and the controls with clustered standard errors below and stars at the 10, 5, and 1 percent levels, the specification rows, and the summary rows (observations, mean of the dependent variable on the estimation sample, the pseudo R² that `ppmlhdfe` reports, and the number of firm clusters). Written as Markdown, LaTeX, and CSV under one file stem; the Markdown is printed with its title, Notes, and Source.


In [8]:
local models col_1 col_2 col_3 col_4
local n_models : word count `models'
local vars c.relevance_z#1.post relevance_z ln_age_l1 ln_at_l1 ln_xrd_l1 rd_missing_l1
local lab_1 "Topic relevance × Post-2018"
local lab_2 "Topic relevance (standard deviations)"
local lab_3 "Log firm age, t − 1"
local lab_4 "Log total assets, t − 1"
local lab_5 "Log of one plus R&D expense, t − 1"
local lab_6 "R&D expense missing, t − 1"
local table_title "Table 2—Poisson Difference-in-Differences Estimates of Yearly Patent Counts on Topic Relevance × Post-2018"
local table_notes "The unit of observation is a firm-year: the T4.1.3 analysis sample (firms of the all-years T3 sample with at least one patent filed in 1999 to 2017), fiscal years 1999 to 2021 with a 10-K and the lagged controls observed. The dependent variable is the number of KPSS patents filed in the year that carry at least one CPC code of the project's privacy code list (T4.1.2) in columns 1 and 2, and all KPSS patents filed in the year in columns 3 and 4. Each column is a Poisson pseudo-maximum-likelihood regression (ppmlhdfe) with NAICS two-digit-by-year fixed effects absorbed, and firm fixed effects in columns 2 and 4; units whose count is zero throughout drop out. Topic relevance is the firm's score, the cosine similarity between a firm-level summary of its 10-K Item 1 sections and the policy summary of legislator statements on consumer data privacy legislation, one score per firm, standardized over the regression firm-years. Post-2018 equals one in fiscal years 2018 to 2021, after the Facebook–Cambridge Analytica revelations of March 2018; its main effect is absorbed by the NAICS-by-year fixed effects, and the score's main effect by the firm fixed effects where present. The interaction is the post-minus-pre change in the log expected count per standard deviation of relevance relative to less relevant firms in the same NAICS code and year. Controls are measured at fiscal year t − 1: firm age is years since the first CRSP listing; R&D means research and development, entered as the log of one plus the expense with missing R&D set to zero and flagged. Standard errors clustered by firm are in parentheses; * p < 0.10, ** p < 0.05, *** p < 0.01. Pseudo R² is the one ppmlhdfe reports."
local table_source "KPSS extended patent data (Kogan, Papanikolaou, Seru, and Stoffman, 2017, updated to 2025) with CPC codes; the privacy CPC code list in privacy_cpc/privacy_cpc_codes.xlsx; SEC EDGAR 10-K filings; congressional hearing statements from GovInfo; Compustat; CRSP-Compustat link; authors' calculations."
local table_name "table_2_ppml_did_patents_relevance_post2018"

quietly {
preserve
clear
set obs 1
gen str80 label = "Dependent variable"
forvalues k = 1/4 {
    gen str40 c`k' = ""
}
local k = 0
foreach m of local models {
    local ++k
    estimates restore `m'
    replace c`k' = "`e(dv_label)'" in 1
}
local j = 0
foreach v of local vars {
    local ++j
    local r = _N + 1
    set obs `=_N + 2'
    replace label = "`lab_`j''" in `r'
    local k = 0
    foreach m of local models {
        local ++k
        estimates restore `m'
        local col = colnumb(e(b), "`v'")
        if `col' < . & _se[`v'] > 0 {
            local b = _b[`v']
            local se = _se[`v']
            local p = 2 * normal(-abs(`b' / `se'))
            local stars = cond(`p' < 0.01, "***", cond(`p' < 0.05, "**", cond(`p' < 0.10, "*", "")))
            local bs = subinstr(string(`b', "%9.3f"), "-", "−", 1)
            local ses = subinstr(string(`se', "%9.3f"), "-", "−", 1)
            replace c`k' = "`bs'`stars'" in `r'
            replace c`k' = "(`ses')" in `=`r' + 1'
        }
    }
}
foreach row in "Lagged controls|controls" "Firm fixed effects|firm_fe" "NAICS two-digit-by-year fixed effects|naics2_year_fe" {
    local text = substr("`row'", 1, strpos("`row'", "|") - 1)
    local macro = substr("`row'", strpos("`row'", "|") + 1, .)
    local r = _N + 1
    set obs `r'
    replace label = "`text'" in `r'
    local k = 0
    foreach m of local models {
        local ++k
        estimates restore `m'
        replace c`k' = "`e(`macro')'" in `r'
    }
}
foreach row in "Observations|N|%12.0fc" "Mean of dependent variable|mean_dv|%9.3f" "Pseudo R²|r2_p|%9.2f" "Number of clusters (firms)|clusters|%12.0fc" {
    tokenize "`row'", parse("|")
    local text "`1'"
    local scalar "`3'"
    local format "`5'"
    local r = _N + 1
    set obs `r'
    replace label = "`text'" in `r'
    local k = 0
    foreach m of local models {
        local ++k
        estimates restore `m'
        replace c`k' = trim(string(e(`scalar'), "`format'")) in `r'
    }
}

* Markdown
file open md using "${TABLE_FOLDER}/`table_name'.md", write replace text
file write md "**`table_title'**" _n _n
file write md "| | (1) | (2) | (3) | (4) |" _n
file write md "|:---|:---:|:---:|:---:|:---:|" _n
forvalues i = 1/`=_N' {
    file write md "| `=label[`i']' | `=c1[`i']' | `=c2[`i']' | `=c3[`i']' | `=c4[`i']' |" _n
}
file write md _n "*Notes*: `table_notes'" _n _n "*Source*: `table_source'" _n
file close md

* LaTeX
local title_tex = subinstr("`table_title'", "—", "---", 1)
local notes_tex = subinstr("`table_notes'", "&", "\&", .)
file open tex using "${TABLE_FOLDER}/`table_name'.tex", write replace text
file write tex "\begin{table}[htbp]" _n "\centering" _n "\textbf{`title_tex'}\par" _n
file write tex "\begin{tabular}{lcccc}" _n "\toprule" _n " & (1) & (2) & (3) & (4) \\" _n "\midrule" _n
forvalues i = 1/`=_N' {
    local l = label[`i']
    local l = subinstr("`l'", "&", "\&", .)
    local l = subinstr("`l'", "R²", "R\textsuperscript{2}", 1)
    if "`l'" == "Lagged controls" | "`l'" == "Observations" {
        file write tex "\midrule" _n
    }
    file write tex "`l' & `=c1[`i']' & `=c2[`i']' & `=c3[`i']' & `=c4[`i']' \\" _n
}
file write tex "\bottomrule" _n "\end{tabular}" _n
file write tex "\par\textit{Notes}: `notes_tex'" _n
file write tex "\par\textit{Source}: `table_source'" _n "\end{table}" _n
file close tex

* CSV
insobs 1, before(1)
replace label = "" in 1
forvalues k = 1/4 {
    replace c`k' = "(`k')" in 1
}
export delimited label c1 c2 c3 c4 using "${TABLE_FOLDER}/`table_name'.csv", novarnames replace
restore
}

type "${TABLE_FOLDER}/`table_name'.md"
display _n "Saved: ${TABLE_FOLDER}/`table_name'.md, .tex, and .csv"


(Note: Below code run with echo to enable preserve/restore functionality.)

















**Table 2—Poisson Difference-in-Differences Estimates of Yearly Patent Counts on Topic Relevance × Post-2018**

| | (1) | (2) | (3) | (4) |
|:---|:---:|:---:|:---:|:---:|
| Dependent variable | Privacy patents | Privacy patents | All patents | All patents |
| Topic relevance × Post-2018 | −0.134 | −0.095 | −0.124 | −0.001 |
|  | (0.102) | (0.067) | (0.077) | (0.052) |
| Topic relevance (standard deviations) | 0.523*** |  | 0.211* |  |
|  | (0.183) |  | (0.123) |  |
| Log firm age, t − 1 | 0.256 | 0.048 | 0.292* | −0.035 |
|  | (0.320) | (0.181) | (0.172) | (0.073) |
| Log total assets, t − 1 | 0.376*** | 0.180 | 0.250*** | 0.193** |
|  | (0.093) | (0.114) | (0.061) | (0.085) |
| Log of one plus R&D expense, t − 1 | 0.514*** | 0.045 | 0.573*** | 0.226*** |
|  | (0.132) | (0.105) | (0.089) | (0.077) |
| R&D expense missing, t − 1 | 1.844** | −0.346 | 1.763*** | 1.027*** |
|  | (0.874) | (0.586) | (0.538) | (0.374) |
| Lagged controls | Yes | Yes | Yes | Yes |
| Firm fixed effects | No |